# reBAP EDA — the German imbalance price

**Data:** `data/rebap_2022-2026.csv` (netztransparenz.de, reBAP / NrvSaldo, quarter-hourly, EUR/MWh) ·
**Reference structure:** [`team-EDA.ipynb`](team-EDA.ipynb) (spec 03)

## What this notebook is

A **personal exploration** (`EDA-<name>` track) of the reBAP price series — the
*regelzonenübergreifender einheitlicher Bilanzausgleichsenergiepreis*, the uniform imbalance
price that balancing-responsible parties pay or receive per MWh of imbalance. The project's
CLAUDE.md reserves reBAP for cost calculation *after* modelling and keeps it out of the team
EDA; this notebook does not change that. Its conclusions are one member's reading of the data,
written as proposals, not project facts.

It mirrors the **applicable** sections of `team-EDA.ipynb` and skips the ones with no reBAP
counterpart (solar/wind, holidays, Euro 2024, harmonics). It is **standalone**: `data/smard.csv`
is not loaded, and nothing here relates prices to load.

## Sections

| # | Section | Mirrors |
|---|---|---|
| 1 | Setup — loading, helpers, unambiguous index, `YEARS`, `LOADED` | team §1 |
| 2 | Sanity check | team §2 |
| 3 | Univariate and time structure | team §3.1–3.5, 3.7–3.8 |
| 4 | Temporal dependence, decomposition, ramps | team §6.1–6.2, 6.4–6.5 |
| 5 | Context appendix, findings, self-check | team §7.1, 7.3, Findings |

## Conventions

- The shared frame is **`time_series`**, indexed by **tz-aware `Europe/Berlin`** timestamps at
  the native **15-minute** resolution. `SERIES` names the two price columns, `DERIVED` the
  calendar columns; the split keeps `.describe()` and `.corr()` honest.
- Units: **EUR/MWh** on every y-axis. Ramps are EUR/MWh per 15-minute step.
- Weeks are ISO (Monday start); seasons meteorological with `season_year = year + (month == 12)`.
- **No thresholds.** Tails are selected by rank, inside their own plotting cell, for description
  only. No boolean flag or label column is created anywhere.
- Helpers are **copied** from `team-EDA.ipynb`; the two deliberate changes are stated in §1.1.
- No literal calendar year appears in code — everything year-dependent derives from `YEARS`.

---

## 1 — Setup

`data/` is gitignored, so the file is not in a fresh clone. `notebooks/API-connection.ipynb`
Part 2 regenerates it — it needs OAuth2 credentials in a gitignored `.env` and skips cleanly
without them. Note the filename: CLAUDE.md refers to `data/rebap.csv`; the file the pipeline
actually wrote is `rebap_2022-2026.csv`, and that is what is read here.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.rcParams["figure.max_open_warning"] = 0

# Walk up from the working directory to the first parent holding a `data/` folder, so the
# notebook runs unmodified from notebooks/01_eda/ (Jupyter) or from the repo root (nbconvert).
DATA_DIR = next(
    (p / "data" for p in (Path.cwd(), *Path.cwd().parents) if (p / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(
        f"no data/ directory found in {Path.cwd()} or any parent — start the kernel inside the "
        "repository, then re-run."
    )
DATA = DATA_DIR / "rebap_2022-2026.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} not found. data/ is gitignored, so the file is not in a fresh clone — "
        "regenerate it by running Part 2 of notebooks/API-connection.ipynb (needs .env credentials)."
    )

# One series, one colour (team-EDA's navy). Grey for reference lines.
PRICE_COLOR, MUTED = "#17365D", "#707B8C"

# The one resolution constant. Every "one step" in this notebook is one of these.
RESOLUTION = pd.Timedelta("15min")
STEPS_PER_HOUR = int(pd.Timedelta("1h") / RESOLUTION)
STEPS_PER_DAY = 24 * STEPS_PER_HOUR
STEPS_PER_WEEK = 7 * STEPS_PER_DAY

print(f"pandas {pd.__version__} · numpy {np.__version__} · seaborn {sns.__version__}")
print(f"reading {DATA}")
print(f"resolution {RESOLUTION} -> {STEPS_PER_HOUR}/h, {STEPS_PER_DAY}/day, {STEPS_PER_WEEK}/week")

### 1.1 — Helpers

`_complete_periods`, `period_mean`, `style_timeseries` and `seasonal_plot` are **copied** from
`team-EDA.ipynb`. `period_energy` is not: it converts MWh sums into MW and MWh/day, which has no
meaning for a price.

Two deliberate changes, both forced by this dataset rather than chosen:

1. **Closing edge.** The original closes the last observation with `+ 1h`, because SMARD rows
   are hours. Here a row is a quarter-hour, so the edge is `+ RESOLUTION`. Without this the last
   complete week of the record would be dropped as "incomplete".
2. **Wall clock.** The index is tz-aware (§1.3 explains why). `to_period()` drops the timezone
   and pandas then refuses to compare naive period edges with tz-aware bounds, so both helpers
   first take the local wall-clock view via `wall_clock()`. Grouping by wall-clock period is also
   the *right* thing: both occurrences of the autumn fold hour belong to the same week and month.

`wall_clock()` is the only new helper. It is used wherever a plot needs local time on its x-axis,
because matplotlib would otherwise silently draw tz-aware timestamps in UTC.

In [ ]:
def wall_clock(index):
    """Local wall-clock view of a DatetimeIndex: tz-aware -> naive local, naive -> unchanged."""
    return index.tz_localize(None) if index.tz is not None else index


def _complete_periods(index, freq):
    """The calendar periods of `freq` that `index` covers completely.

    The one edge rule of this notebook, in one place. A period counts only if it starts no
    earlier than the first observation and ends no later than the last observation's closing
    edge. `period_mean` defers to this.

    Copied from team-EDA.ipynb; closing edge is RESOLUTION (not 1h) and the comparison is made
    on the wall clock — see §1.1.
    """
    index = wall_clock(index)
    periods = index.to_period(freq).unique().sort_values()
    complete = (periods.start_time >= index.min()) & (
        periods.end_time <= index.max() + RESOLUTION
    )
    return periods[complete]


def period_mean(series, freq):
    """Mean of `series` per calendar period (`"W"`, `"M"`, ...), indexed by period start.

    Periods that the data does not cover completely are dropped, so the edges of a plot are not
    partial-period artefacts. Note the rule is "not fully covered", not "the first and the last".

    Copied from team-EDA.ipynb; groups on the wall clock — see §1.1.
    """
    wall = wall_clock(series.index)
    agg = series.groupby(wall.to_period(freq)).mean()
    agg = agg.loc[_complete_periods(series.index, freq)]
    agg.index = agg.index.start_time
    return agg


def style_timeseries(ax, title, ylabel):
    """Custom grid, no box, year ticks.

    `ylabel` is required: every plot must state its unit. Copied verbatim from team-EDA.ipynb.
    """
    ax.set_title(
        title,
        loc="center",
        fontsize=15,
        pad=12
    )
    ax.set_xlabel("")
    ax.set_ylabel(
        ylabel,
        color="grey"
    )
    ax.grid(
        axis="y",
        color="0.9",
        linewidth=0.8
    )
    ax.set_axisbelow(True)

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    ax.tick_params(
        colors="black",
        length=0  # hide ticks of values
    )
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")


def seasonal_plot(df, y_value, title, ylabel):
    """Creates a seasonal plot from a dataframe. Copied verbatim from team-EDA.ipynb.

    Args:
        df (DataFrame): frame with separate `month` and `year` columns, already aggregated to
            one row per (year, month). Passing raw data makes seaborn bootstrap a confidence
            interval per cell over tens of thousands of rows — minutes of runtime, meaningless
            band.
        y_value (str): name of the y-value to plot
        title (str): title of the plot
        ylabel (str): axis description, including units. Required, and actually applied.
    """
    fig, ax = plt.subplots(figsize=(14, 5))

    sns.lineplot(
        data=df,
        x="month",
        y=y_value,
        hue="year",
        palette="viridis",
        legend=True,
        ax=ax
    )
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(range(1, 13))
    plt.tight_layout()
    plt.show()


print("helpers defined: wall_clock, _complete_periods, period_mean, style_timeseries, seasonal_plot")

### 1.2 — Load and prepare

Same German Excel CSV format as SMARD (`;` separator, `,` decimal, `utf-8-sig`), so every price
column arrives as text. Seven columns come off disk; three of them (`Datenkategorie`,
`Datentyp`, `Einheit`) are metadata that should be constant across the whole file — asserted,
then dropped. `bis_utc` is kept until §1.3, where it earns its place.

The two price columns keep their German names, prefixed for snake_case:
`reBAP unterdeckt` → `rebap_unterdeckt` (price when the balancing group is *short*) and
`reBAP ueberdeckt` → `rebap_ueberdeckt` (price when it is *long*). Whether they differ is
checked rather than assumed.

In [ ]:
# The CSV headers exactly as notebooks/API-connection.ipynb Part 2 writes them.
COLUMNS = {
    "reBAP unterdeckt": "rebap_unterdeckt",
    "reBAP ueberdeckt": "rebap_ueberdeckt",
}
META_COLUMNS = ["Datenkategorie", "Datentyp", "Einheit"]

raw = pd.read_csv(DATA, delimiter=";", encoding="utf-8-sig")

expected = {"timestamp", "bis_utc"} | set(META_COLUMNS) | set(COLUMNS)
assert set(raw.columns) == expected, (
    f"unexpected CSV header: {sorted(set(raw.columns) ^ expected)}"
)

print("as read from disk — note the comma decimals and the string dtypes:")
display(raw.head(3))
display(raw.dtypes.to_frame("dtype"))

In [ ]:
raw = raw.rename(columns=COLUMNS)

for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)

# The metadata columns must be constant. Print the single value each holds, then drop them.
for col in META_COLUMNS:
    values = raw[col].unique()
    assert len(values) == 1, f"{col} is not constant: {values}"
    print(f"{col:<15} = {values[0]!r}  (constant, dropped)")
UNIT = raw["Einheit"].iloc[0]
raw = raw.drop(columns=META_COLUMNS)

# Are the two prices ever different? Report the distribution of the difference, not a yes/no,
# so a rounding artefact could not be mistaken for a real split.
diff = raw["rebap_unterdeckt"] - raw["rebap_ueberdeckt"]
print(f"\nrebap_unterdeckt - rebap_ueberdeckt: "
      f"{int((diff != 0).sum())} non-zero rows of {len(raw):,}, max |diff| = {diff.abs().max()}")
assert (diff == 0).all(), "the two reBAP columns differ — the notebook assumes one price"

# The analysed price. Both columns stay in the frame; every plot below uses this one.
PRICE = "rebap_unterdeckt"
print(f"\nPRICE = {PRICE!r}, unit {UNIT}")

**The two columns are identical on every row.** Since the single-price regime, `unterdeckt` and
`ueberdeckt` are the same number by construction; the file carries it twice. The assertion above
makes that a checked fact rather than a footnote, and everything downstream analyses
`rebap_unterdeckt` alone. If a later fetch ever reaches into a period where the two diverged,
the assertion fails loudly instead of the notebook silently showing half the picture.

### 1.3 — An unambiguous index from `timestamp` + `bis_utc`

`timestamp` is the interval **start in local time** (`Europe/Berlin`). On the last Sunday of
October the clocks go back at 03:00 CEST → 02:00 CET, so the local quarter-hours 02:00, 02:15,
02:30 and 02:45 each **occur twice**. SMARD collapsed that fold upstream; netztransparenz.de did
not — both occurrences are present, with different prices. As a naive local index that is 16
duplicate timestamps and no way to tell them apart.

`bis_utc` — the interval **end in UTC**, time of day only — resolves it: during CEST the local
start is 2 h ahead of UTC, during CET 1 h. Subtracting the UTC end from the local end yields the
offset for every row, and that offset is exactly what `tz_localize(ambiguous=...)` needs. The
result is a tz-aware index that is unique, keeps every row, and still reads as local time in
`.hour`, `.dayofweek` and the calendar columns of §1.4.

In [ ]:
local_start = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")
local_end = local_start + RESOLUTION

# UTC end-of-interval as a naive timestamp on the same calendar date as the local end. The
# date is approximate for the few rows where UTC and local dates differ (around midnight);
# the modulo below removes that.
utc_end_time = pd.to_timedelta(raw["bis_utc"] + ":00")
utc_end_same_day = local_end.dt.normalize() + utc_end_time

# Offset local - UTC, folded into [0, 24h). Must be exactly 1h (CET) or 2h (CEST) everywhere.
offset = (local_end - utc_end_same_day) % pd.Timedelta("24h")
offset_hours = offset.dt.total_seconds() / 3600
counts = offset_hours.value_counts().sort_index()
print("local - UTC offset (hours) and row counts:")
print(counts.to_string())
assert set(counts.index) <= {1.0, 2.0}, "offsets other than CET/CEST found"

is_dst = (offset_hours == 2.0).to_numpy()

# `ambiguous` takes a boolean per row: True = the DST (CEST) occurrence. `nonexistent="raise"`
# proves the spring-forward gap holds no rows — a row at a non-existent 02:xx would fail here.
index = pd.DatetimeIndex(local_start).tz_localize(
    "Europe/Berlin", ambiguous=is_dst, nonexistent="raise"
)

# Round trip: the reconstructed index, converted back to UTC and closed, must reproduce bis_utc
# on EVERY row — including the 16 fold rows a naive parse could not distinguish.
reconstructed = (index + RESOLUTION).tz_convert("UTC").strftime("%H:%M")
mismatch = int((reconstructed != raw["bis_utc"].to_numpy()).sum())
print(f"\nround trip index -> UTC end: {mismatch} mismatches of {len(index):,}")
assert mismatch == 0

time_series = raw.drop(columns=["timestamp", "bis_utc"]).set_index(index).sort_index()
time_series.index.name = "timestamp"
del raw, local_start, local_end, utc_end_time, utc_end_same_day, offset, offset_hours, index, reconstructed

print(f"index tz     : {time_series.index.tz}")
print(f"index unique : {time_series.index.is_unique}, monotonic: {time_series.index.is_monotonic_increasing}")
time_series.head(3)

In [ ]:
# The fold, made visible: the first autumn switch in the record, 01:45 -> 03:00 local. Each 02:xx
# appears twice, first as +02:00 (CEST), then as +01:00 (CET), with different prices.
wall = wall_clock(time_series.index)
autumn_switches = pd.DatetimeIndex(
    [pd.date_range(f"{y}-10-01", f"{y}-10-31", freq="W-SUN")[-1] for y in wall.year.unique()]
)
autumn_switches = autumn_switches[(autumn_switches >= wall.min()) & (autumn_switches <= wall.max())]
first_fold = autumn_switches[0]

fold_rows = time_series.loc[
    f"{first_fold:%Y-%m-%d} 01:45":f"{first_fold:%Y-%m-%d} 03:00", [PRICE]
]
fold_rows = fold_rows.assign(utc_offset=fold_rows.index.strftime("%z"))
print(f"{len(autumn_switches)} autumn switches in the record: {[d.date().isoformat() for d in autumn_switches]}")
print(f"\n{first_fold:%Y-%m-%d}, hour by hour — 02:00..02:45 occur twice:")
display(fold_rows)
assert len(fold_rows) == 2 * 4 + 2, len(fold_rows)  # 01:45, 4x02:xx twice, 03:00

### 1.4 — Derived columns, `SERIES` / `DERIVED`, and `YEARS`

All derived columns are defined here, in one place. `SERIES` names the two price columns and
`DERIVED` the calendar columns; the assertion that the frame holds *exactly* those keeps any
later cell from quietly adding a flag. Calendar columns are taken from the tz-aware index, so
`hour` and `dow` are **local** — the fold hour has eight rows at `hour == 2` on each switch day,
which is the truth of a 25-hour day, not a defect.

`spans_gap` is `True` on the row following a jump larger than one step. With a tz-aware index
the autumn fold is *not* a jump (real elapsed time is continuous), so only the spring-forward
gaps can trigger it. §2 counts them.

In [ ]:
SERIES = list(COLUMNS.values())

DAY_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Meteorological seasons, with December assigned to the FOLLOWING year's winter.
SEASON_OF_MONTH = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]

time_series["year"] = time_series.index.year
time_series["month"] = time_series.index.month
time_series["hour"] = time_series.index.hour
time_series["dow"] = time_series.index.dayofweek
time_series["is_weekend"] = time_series.index.dayofweek >= 5
time_series["date"] = time_series.index.date
time_series["season"] = pd.Categorical(
    time_series.index.month.map(SEASON_OF_MONTH), categories=SEASON_ORDER, ordered=True
)
time_series["season_year"] = time_series.index.year + (time_series.index.month == 12)
# True on the row FOLLOWING a gap. The first row is False (NaT comparison), not NaN.
time_series["spans_gap"] = time_series.index.to_series().diff() > RESOLUTION

DERIVED = [
    "year", "month", "hour", "dow", "is_weekend",
    "date", "season", "season_year", "spans_gap",
]

assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

# Plain ints, not np.int32: they end up in titles, labels and dict keys all over the notebook.
YEARS = sorted(int(y) for y in time_series["year"].unique())

# Snapshot for §5's closing self-check, taken before any other cell can touch the frame.
LOADED = {
    "rows": len(time_series),
    "start": time_series.index.min(),
    "end": time_series.index.max(),
}

print(f"{len(SERIES)} data columns + {len(DERIVED)} derived = {time_series.shape[1]} columns")
print(f"YEARS  = {YEARS}")
print(f"LOADED = {LOADED}")
display(time_series[SERIES].describe().T)
time_series[DERIVED].head(3)

### 1.5 — Initial self-check

In [ ]:
assert all(pd.api.types.is_float_dtype(time_series[c]) for c in SERIES), time_series[SERIES].dtypes
assert time_series.index.tz is not None and str(time_series.index.tz) == "Europe/Berlin"
assert time_series.index.is_monotonic_increasing, "index is not sorted"
assert time_series.index.is_unique, "index has duplicate timestamps"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert YEARS == sorted(int(y) for y in time_series["year"].unique())
assert (time_series[SERIES[0]] == time_series[SERIES[1]]).all()

print("setup self-check passed")
print(f"  {len(SERIES)} float series (identical), tz-aware index sorted and unique")
print(f"  columns == SERIES + DERIVED ({len(SERIES) + len(DERIVED)} columns)")
print(f"  {LOADED['rows']:,} rows, {LOADED['start']} -> {LOADED['end']}, years {YEARS}")

---

## 2 — Sanity check

Mirrors team §2: a deliberately small audit — coverage, gaps, duplicates, missing values, value
ranges — followed by a one-week plausibility plot. The governing rule is the team's: **nothing is
repaired.** Gaps are found and named, extremes are listed, nothing is filled, clipped or dropped.

The checks differ from the SMARD ones where the data differs. A price has no physical sign
constraint (negative imbalance prices are real and meaningful), so "value ranges" here means
*where the tails are and whether anything looks like a bound or a stuck sensor*, not "is it
below zero".

### 2.1 — Coverage, gaps, duplicates, missing values

With a tz-aware index the complete grid is built in **absolute time**. In that view an autumn
fold is nothing at all (real time is continuous through it, and §1.3 kept both occurrences), and
a spring-forward switch is only a gap if the source actually skipped the four real quarter-hours
between 01:00 and 02:00 UTC. Whether it did is what the count below answers.

In [ ]:
full_index = pd.date_range(time_series.index.min(), time_series.index.max(), freq=RESOLUTION)
missing_slots = full_index.difference(time_series.index)

missing = pd.DataFrame(
    {
        "n_missing": time_series[SERIES].isna().sum(),
        "share_%": (time_series[SERIES].isna().mean() * 100).round(3),
    }
)
display(missing)

n_duplicates = int(time_series.index.duplicated().sum())
print(f"missing values across all {len(SERIES)} series : {int(missing['n_missing'].sum())}")
print(f"duplicate timestamps                     : {n_duplicates}")
print(f"rows in file                             : {len(time_series):,}")
print(f"complete 15-min index, absolute time     : {len(full_index):,}")
print(f"missing slots, absolute time             : {len(missing_slots):,}")
print(f"rows flagged spans_gap                   : {int(time_series['spans_gap'].sum())}")

# `spans_gap` marks the row FOLLOWING each gap; the two views must agree.
N_GAPS = len(missing_slots)
assert N_GAPS == 0 or int(time_series["spans_gap"].sum()) > 0
assert (N_GAPS > 0) == bool(time_series["spans_gap"].any())

# The contrast that explains why a naive check "finds" gaps: on the local WALL CLOCK the hour
# 02:00-02:59 of each spring switch does not exist, so a naive hourly grid invents four slots per
# switch that no data could ever fill. Listed here so nobody re-discovers them as missing data.
wall = wall_clock(time_series.index)
phantom = pd.date_range(wall.min(), wall.max(), freq=RESOLUTION).difference(wall)
phantom_days = (
    pd.Series(phantom.normalize().date, name="date").value_counts().sort_index()
    .rename("phantom_slots").to_frame()
)
phantom_days["weekday"] = pd.DatetimeIndex(phantom_days.index).day_name()
print(f"\nnaive wall-clock grid, for contrast: {len(phantom)} slots absent, "
      f"all labelled {sorted(set(phantom.hour))}:xx local — the spring switch days:")
display(phantom_days)
assert set(phantom.hour) <= {2}, "a wall-clock phantom outside the 02:xx spring hour"

**Zero missing values, zero duplicate timestamps, zero gaps.** In absolute time the record is
**complete**: 164,156 quarter-hours of real time, each present exactly once. Neither DST switch
costs a row — the autumn fold was preserved by the source (§1.3), and the four real quarter-hours
under the spring switch are present too; they just carry a 03:xx local label instead of a 02:xx
one.

This is a genuine difference from SMARD, and it is worth stating precisely because the intuition
from the team notebook ("five gaps, one per spring switch") does **not** carry over. SMARD's
hourly file lacks the spring 02:00 row *and* collapses the autumn fold, so it loses one hour per
switch in both directions. The reBAP feed loses none. The only thing that looks like a gap is the
wall-clock phantom above — four 02:xx slots per spring switch that a naive local grid invents
and that no data source could ever fill, because that local hour never happened.

Two consequences for the rest of the notebook: `spans_gap` is all-`False` and stays in the frame
only for parity with the team convention; and `.shift()`, `.diff()` and the ACF lag axis are
**exact** here — one step is always exactly fifteen minutes of real time, with no DST caveat.

### 2.2 — Value ranges and the two extremes

No sign check — a negative imbalance price is legitimate. Instead: how much of the record is
negative or exactly zero, where the two extreme values sit, and whether the top of the range is
a bound (many rows at the same ceiling) or a single spike. The last check — runs of identical
consecutive prices — is the price-data analogue of the team's night-solar test: a market price
that does not move for many consecutive quarter-hours is either a real flat period or a stuck
feed, and the run lengths tell which is plausible.

Slices below are **rank-based and descriptive** (§ conventions): nothing is persisted.

In [ ]:
price = time_series[PRICE]

share = lambda mask: f"{int(mask.sum()):>7,} rows  ({100 * mask.mean():5.2f} %)"
print(f"{PRICE} in {UNIT}")
print(f"  negative (< 0)     : {share(price < 0)}")
print(f"  exactly zero       : {share(price == 0)}")
print(f"  positive (> 0)     : {share(price > 0)}")
print(f"  min                : {price.min():>10,.2f}   at {wall_clock(pd.DatetimeIndex([price.idxmin()]))[0]:%Y-%m-%d %H:%M %a}")
print(f"  max                : {price.max():>10,.2f}   at {wall_clock(pd.DatetimeIndex([price.idxmax()]))[0]:%Y-%m-%d %H:%M %a}")

display(
    price.describe(percentiles=[0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999])
    .to_frame(PRICE).T.round(2)
)

# Is the maximum a ceiling or a spike? Count rows at, and within 1 % of, the maximum; same at
# the minimum. Reported, not interpreted as a regulatory cap — the data cannot establish that.
for label, extreme, side in (("max", price.max(), 1), ("min", price.min(), -1)):
    at = int((price == extreme).sum())
    near = int(((extreme - price) * side <= 0.01 * abs(extreme)).sum())
    print(f"  rows exactly at {label} {extreme:>10,.2f}: {at:>4}   within 1 % of it: {near:>4}")

TOP_N = 10
print(f"\n{TOP_N} highest and {TOP_N} lowest quarter-hours (rank-based, description only):")
extremes = pd.concat(
    [price.nlargest(TOP_N).rename("highest").reset_index(),
     price.nsmallest(TOP_N).rename("lowest").reset_index()],
    axis=1,
)
extremes.columns = ["timestamp_high", "highest", "timestamp_low", "lowest"]
for col in ("timestamp_high", "timestamp_low"):
    extremes[col] = wall_clock(pd.DatetimeIndex(extremes[col])).strftime("%Y-%m-%d %H:%M %a")
display(extremes)

In [ ]:
# Runs of identical consecutive prices. A run of length k means the price did not change for
# k quarter-hours. Long runs are listed by timestamp so they can be judged, never removed.
run_id = (price != price.shift()).cumsum()
runs = price.groupby(run_id).agg(start=lambda s: s.index[0], length="size", value="first")
runs["start"] = wall_clock(pd.DatetimeIndex(runs["start"]))

print("run length of identical consecutive prices — how many runs of each length:")
display(runs["length"].value_counts().sort_index().rename("n_runs").to_frame().T)

LONGEST_N = 10
print(f"\nthe {LONGEST_N} longest runs:")
display(runs.nlargest(LONGEST_N, "length").reset_index(drop=True))

print(f"\nshare of rows inside a run longer than {STEPS_PER_HOUR} steps (one hour): "
      f"{100 * runs.loc[runs['length'] > STEPS_PER_HOUR, 'length'].sum() / len(price):.3f} %")

### 2.3 — One ordinary week, end to end

Before summarising nearly five years, look at one seven-day window and ask whether the series
behaves like a market price: it should move every quarter-hour, swing around a positive level,
and occasionally dip below zero.

The week is found **programmatically**, as in the team notebook: the first Monday 00:00 →
Sunday 23:45 window with `STEPS_PER_WEEK` consecutive rows and no missing value. The check is
kept even though §2.1 showed the record is gap-free — it costs nothing, and it is what makes the
selection rule survive a re-fetch that starts on a different day or contains a real gap. Because
the grid is built in absolute time, a DST week is *not* disqualified: it has 672 real
quarter-hours like any other, with a 23- or 25-hour Sunday on the wall clock.

In [ ]:
valid = time_series[SERIES].notna().all(axis=1)
candidate_starts = time_series.index[
    (time_series["dow"] == 0) & (time_series["hour"] == 0) & (time_series.index.minute == 0)
]

week_start, skipped = None, 0
for start in candidate_starts:
    expected = pd.date_range(start, periods=STEPS_PER_WEEK, freq=RESOLUTION)
    if not expected.isin(time_series.index).all() or not valid.loc[expected].all():
        skipped += 1
        continue
    week_start = start
    break

if week_start is None:
    raise RuntimeError("no fully valid Monday-to-Sunday week found")

week_index = pd.date_range(week_start, periods=STEPS_PER_WEEK, freq=RESOLUTION)
week = time_series.loc[week_index]

print(f"first row in the record   : {time_series.index.min()} ({time_series.index.min():%A})")
print(f"candidate Mondays skipped : {skipped}")
print(f"week selected             : {week.index[0]:%a %Y-%m-%d %H:%M} -> {week.index[-1]:%a %Y-%m-%d %H:%M}")
print(f"rows                      : {len(week)}")

In [ ]:
def plot_week(frame, title):
    """One week of quarter-hourly prices, local time on x. Returns the plotted span."""
    fig, ax = plt.subplots(figsize=(13.5, 5.5))
    x = wall_clock(frame.index)

    ax.plot(x, frame[PRICE].to_numpy(), color=PRICE_COLOR, linewidth=1.4, label=PRICE)
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_title(title, fontsize=15, pad=12)
    ax.set_ylabel(UNIT, color="grey")
    ax.set_xlabel("Europe/Berlin time", labelpad=8)
    ax.grid(axis="y", color="0.9", lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a\n%d %b"))
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.2))
    plt.tight_layout()
    plt.show()

    return frame.index[0], frame.index[-1]


week_span = plot_week(
    week, f"reBAP, {week.index[0]:%d %b %Y} – {week.index[-1]:%d %b %Y}, quarter-hourly"
)

print(f"in this week: mean {week[PRICE].mean():,.1f}, median {week[PRICE].median():,.1f}, "
      f"min {week[PRICE].min():,.1f}, max {week[PRICE].max():,.1f} {UNIT}; "
      f"{int((week[PRICE] < 0).sum())} of {len(week)} quarter-hours negative")

### 2.4 — Self-check

In [ ]:
assert len(week) == STEPS_PER_WEEK, len(week)
assert week.index[0].dayofweek == 0 and week.index[0].hour == 0, week.index[0]
assert week.index[-1].dayofweek == 6 and week.index[-1].hour == 23 and week.index[-1].minute == 45, week.index[-1]
assert (week.index.to_series().diff().dropna() == RESOLUTION).all(), "internal gap"
assert week[SERIES].notna().all().all(), "NaN inside the selected week"
assert week_span == (week.index[0], week.index[-1])
assert len(time_series) == LOADED["rows"], "rows were added or dropped since loading"
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)

print("sanity-check self-check passed")
print(f"  {N_GAPS} missing slots in absolute time, {n_duplicates} duplicates, "
      f"{len(phantom)} wall-clock phantom slots (spring 02:xx)")
print(f"  {len(week)} consecutive rows, {week.index[0]:%A %H:%M} -> {week.index[-1]:%A %H:%M}")
print(f"  no column added to time_series ({len(time_series.columns)} columns), {len(time_series):,} rows unchanged")

---

## 3 — Univariate and time structure

Mirrors team §3.1–3.5 and §3.7–3.8 for a single series. Every weekly and monthly aggregate goes
through `period_mean`, so partial edge periods never reach a plot.

### 3.1 — The price over the full record

Three resolutions of the same series, stacked on one time axis: every quarter-hour, the daily
mean, and the weekly and monthly means. The top panel is where the spikes live; the bottom one is
where the level lives. Reading them together is the point — a spike that dominates the top panel
can be invisible in the monthly mean, and a level shift that is obvious in the monthly mean is
buried under noise at the top.

In [ ]:
price = time_series[PRICE]
x_all = wall_clock(price.index)

daily_mean = period_mean(price, "D")
weekly_mean = period_mean(price, "W")
monthly_mean = period_mean(price, "M")

fig, axes = plt.subplots(3, 1, figsize=(15, 13), sharex=True)

axes[0].plot(x_all, price.to_numpy(), lw=0.3, color=PRICE_COLOR)
style_timeseries(axes[0], "reBAP, every quarter-hour", UNIT)

axes[1].plot(daily_mean.index, daily_mean.to_numpy(), lw=0.8, color=PRICE_COLOR)
style_timeseries(axes[1], "daily mean", UNIT)

axes[2].plot(weekly_mean.index, weekly_mean.to_numpy(), lw=1.0, color="#4AA3A5",
             label="weekly mean (labelled by its Monday)")
axes[2].plot(monthly_mean.index, monthly_mean.to_numpy(), lw=2.2, color=PRICE_COLOR,
             label="monthly mean (incomplete edge months dropped)")
style_timeseries(axes[2], "weekly and monthly means", UNIT)
axes[2].legend(frameon=False)

for ax in axes:
    ax.axhline(0, color=MUTED, lw=0.8, zorder=0)

plt.tight_layout()
plt.show()

In [ ]:
# Per-year summary. `neg_%` is the share of quarter-hours below zero. 2026 is a partial year.
per_year = price.groupby(time_series["year"]).agg(
    quarter_hours="size", mean="mean", median="median", std="std", min="min", max="max",
    neg_pct=lambda s: 100 * (s < 0).mean(),
)
per_year.index = YEARS
per_year["complete_year"] = per_year["quarter_hours"] >= 365 * STEPS_PER_DAY
display(per_year.round(1))

**One year dominates the record, and it is the first one.** 2022 sits far above everything since:
a mean of about 216 EUR/MWh against 86–108 in every later year, and monthly means peaking near
420 in late summer. After early 2023 the monthly mean settles into a 60–130 band and stays there —
there is a **level shift, not a trend**. Any model trained across the whole record without
accounting for that shift is fitting two different regimes as one.

**The mean and the median disagree most where the spikes are.** In 2022 the mean (216) runs well
above the median (154); by 2025 the two have converged (91 and 88). That gap is a spike-intensity
measure in disguise, and it shrinks — consistent with the standard deviation halving from ~320 in
2022 to ~140–150 in 2025–26.

**The negative share moves independently of the level.** It rises from 16 % (2022) to a peak of
about 20 % in 2024, then falls to 11–12 % — so the cheapest years are not the most often negative,
and the level and the tails are telling different stories. §4.5 revisits this on a matched window.

Note the top panel is the same data as the bottom one. Read alone it would suggest the series is
mostly spikes; read against the monthly line it is clear the spikes sit on a stable base. 2026 is
a partial year (to 6 September) and is flagged as such in the table.

### 3.2 — Distribution of the price

The tails are roughly a hundred times wider than the interquartile range, so no single axis
shows the whole distribution honestly. Three views of the same 164,156 values:

- **A — full range, linear.** Kept as the demonstration of *why* a plain histogram fails here.
- **B — the central 99 %.** The body, at the cost of cutting 1 % of the record off each side;
  the excluded counts are printed in the title so the cut is never silent.
- **C — full range on a symlog axis.** Linear within ±10 EUR/MWh, logarithmic beyond, with
  log-spaced bins. The only view in which the negative tail, the zero bin, the positive body and
  the extreme spikes are visible at once. Bar heights are counts per bin, and bins widen with
  distance from zero, so heights are comparable *within* a decade, not across the axis.

In [ ]:
values = price.to_numpy()
lo, hi = np.percentile(values, [1, 99])
n_below, n_above = int((values < lo).sum()), int((values > hi).sum())

fig, axes = plt.subplots(3, 1, figsize=(15, 13))

axes[0].hist(values, bins=300, color=PRICE_COLOR)
axes[0].set_title("A — full range, linear axis", fontsize=13, pad=8)

body = values[(values >= lo) & (values <= hi)]
axes[1].hist(body, bins=200, color=PRICE_COLOR)
axes[1].set_title(
    f"B — central 99 % ({lo:,.0f} to {hi:,.0f} {UNIT}); "
    f"excluded: {n_below:,} quarter-hours below, {n_above:,} above",
    fontsize=13, pad=8,
)

LINTHRESH = 10  # linear within ±10 EUR/MWh, logarithmic beyond
decades_neg = np.log10(max(-values.min(), LINTHRESH))
decades_pos = np.log10(max(values.max(), LINTHRESH))
log_bins = np.concatenate([
    -np.logspace(decades_neg, np.log10(LINTHRESH), 120),
    np.linspace(-LINTHRESH, LINTHRESH, 9)[1:-1],
    np.logspace(np.log10(LINTHRESH), decades_pos, 120),
])
axes[2].hist(values, bins=log_bins, color=PRICE_COLOR)
axes[2].set_xscale("symlog", linthresh=LINTHRESH)
axes[2].set_title(f"C — full range, symlog axis (linear within ±{LINTHRESH} {UNIT}, log bins)", fontsize=13, pad=8)

for ax in axes:
    ax.axvline(0, color=MUTED, lw=1.0)
    ax.set_xlabel(UNIT)
    ax.set_ylabel("quarter-hours", color="grey")
    ax.grid(axis="y", color="0.9", lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")

plt.tight_layout()
plt.show()

print(f"1st / 99th percentile : {lo:,.2f} / {hi:,.2f} {UNIT}")
print(f"share below zero      : {100 * (values < 0).mean():.2f} %   exactly zero: {int((values == 0).sum())} quarter-hours")

**Panel A is the finding, not a failed plot.** A linear axis over the full range shows a single
needle: the distribution's body occupies well under 1 % of the axis the extremes demand. Any
summary of this series that quotes a mean and a standard deviation is describing something that
almost never happens.

**The body is not one hump.** Panel B resolves the central 99 % into a main mass between roughly
30 and 200 EUR/MWh with two visible modes, a long positive shoulder running out past 700, and a
**distinct spike sitting right at zero**. The zero spike is a real feature of the series rather
than a rounding artefact — §2.2 counted 31 quarter-hours at exactly 0.00, so the spike is not
those; it is the density of near-zero prices.

**The negative side is its own population.** Panel C is the only view that shows this: on a symlog
axis the negative values form a broad mode of their own around −50 to −100 EUR/MWh, separated from
the positive body by the narrow near-zero region. This is a genuine contrast with the team's
residual load, which §6.3 there describes as a single continuum with no second mode — for reBAP,
"negative" looks less like the far end of one distribution and more like a different state of the
system. 15.50 % of quarter-hours are in it.

**Reading caution for panel C.** Bin widths grow with distance from zero, so bar heights compare
honestly within a decade but not across the axis; the apparent bulk of the extreme tails is a
binning effect, and the counts in panel B's title are the reliable figures.

### 3.3 — Annual seasonality

Monthly means, month on the x-axis, one line per year. The last year in the record is partial,
so its line stops early — that is the data ending, not a collapse. For a price the monthly
*mean* is pulled by spikes, so a second panel shows the monthly **median**, which is not.

In [ ]:
# The record's own end month, so no calendar literal is needed in the title.
last_complete = period_mean(price, "M").index.max()

monthly = period_mean(price, "M").to_frame(PRICE)
monthly["year"] = monthly.index.year
monthly["month"] = monthly.index.month
seasonal_plot(
    monthly, PRICE,
    f"reBAP: monthly mean by month of year, coloured by year (last full month {last_complete:%b %Y})",
    UNIT,
)

# Monthly median, built the same way as period_mean but with .median(); same complete-period rule.
wall = wall_clock(price.index)
monthly_median = price.groupby(wall.to_period("M")).median()
monthly_median = monthly_median.loc[_complete_periods(price.index, "M")]
monthly_median.index = monthly_median.index.start_time
monthly_median = monthly_median.to_frame(PRICE)
monthly_median["year"] = monthly_median.index.year
monthly_median["month"] = monthly_median.index.month
seasonal_plot(
    monthly_median, PRICE,
    "reBAP: monthly MEDIAN by month of year, coloured by year",
    UNIT,
)

**There is no annual cycle in the price level worth speaking of.** Pooled across all years the
month-of-year median moves only between about 82 and 114 EUR/MWh — a spread smaller than the
gap between a weekday and a weekend (§3.4), and far smaller than the year-to-year level shift of
§3.1. The curve has no winter-high/summer-low shape.

This is a sharp contrast with the team's §3.3, where grid load traces a clean seasonal U and the
year lines sit neatly stacked by level. Here the year lines **cross each other repeatedly**: the
ordering of the years changes from month to month, which is what "no stable seasonal shape" looks
like on this plot.

**The mean and median panels tell different stories, and the median is the trustworthy one.** In
the mean panel 2022 towers over everything, peaking in July–August; in the median panel that peak
is much smaller and the other years are legible. August 2022 has a median of 523 EUR/MWh against
72 a month later — that is the energy crisis passing through, an event in time rather than a
recurring August effect, and it is the single feature most likely to be mistaken for seasonality.

### 3.4 — Weekly rhythm

Mean and median price by day of week, and the weekday/weekend contrast as a table. Unlike grid
load, a price has no reason to follow the working week *directly* — but the imbalance it prices
is driven by forecast errors in load and renewables, and those do have a weekly structure.

In [ ]:
dow_profile = price.groupby(time_series["dow"]).agg(mean="mean", median="median")
dow_profile.index = DAY_NAMES

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(dow_profile.index, dow_profile["mean"].to_numpy(), marker="o", color=PRICE_COLOR, lw=2, label="mean")
ax.plot(dow_profile.index, dow_profile["median"].to_numpy(), marker="o", color=PRICE_COLOR, lw=1.4,
        linestyle="--", label="median")
ax.set_title("reBAP level by day of week", fontsize=15, pad=12)
ax.set_ylabel(UNIT, color="grey")
ax.set_xlabel("day of week")
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

weekend_contrast = price.groupby(time_series["is_weekend"]).agg(
    mean="mean", median="median", std="std", neg_pct=lambda s: 100 * (s < 0).mean()
).T
weekend_contrast.columns = ["weekday", "weekend"]
weekend_contrast["delta_%"] = (100 * (weekend_contrast["weekend"] / weekend_contrast["weekday"] - 1)).round(1)
display(weekend_contrast.round(1))

**The weekend is cheaper, but the interesting part is not the level.** The mean falls 30 % from
weekday to weekend and the median only 19 % — the smaller median move says a good part of the mean
gap is spikes, not a shifted level.

**The real weekend effect is in the negative tail.** The share of quarter-hours below zero rises
from 12.8 % on weekdays to 22.1 % at weekends — a **72 % relative increase**, by far the largest
weekday/weekend contrast in the table. Weekends do not simply shift the price down; they make the
oversupply state much more likely.

The mechanism is the same one the team describes in its §3.4, arriving here one step removed:
demand falls at weekends while wind and solar do not care what day it is, so the system is more
often long. What reBAP adds is that this shows up in the price as a **tail probability** rather
than as a level — which is the recurring theme of this notebook.

### 3.5 — Daily rhythm, per season and day type

Mean price by **quarter-hour of day** (96 points, local time), one panel per season, a weekday
and a weekend line in each. Solid is the mean, dashed the median — where they separate, spikes
are doing the work.

In [ ]:
qh = time_series["hour"] + time_series.index.minute / 60  # quarter-hour of day, in hours
groups = [time_series["season"], time_series["is_weekend"], qh.rename("qh")]
qh_profile = price.groupby(groups, observed=True).agg(mean="mean", median="median")

DAY_TYPE_COLOR = {"weekday": "#E95D0F", "weekend": "#2C6EBA"}

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True, sharex=True)

for ax, season in zip(axes.flat, SEASON_ORDER):
    for weekend, label in [(False, "weekday"), (True, "weekend")]:
        profile = qh_profile.loc[(season, weekend)]
        ax.plot(profile.index, profile["mean"].to_numpy(), color=DAY_TYPE_COLOR[label], lw=1.8,
                label=f"{label} mean")
        ax.plot(profile.index, profile["median"].to_numpy(), color=DAY_TYPE_COLOR[label], lw=1.2,
                linestyle="--", label=f"{label} median")
    ax.axhline(0, color=MUTED, lw=0.8)
    ax.set_title(season, fontsize=13, pad=8)
    ax.set_xlabel("hour of day (Europe/Berlin)")
    ax.set_ylabel(UNIT, color="grey")
    ax.set_xticks(range(0, 24, 3))
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
    ax.legend(frameon=False, fontsize=8, ncol=2)

fig.suptitle("reBAP by quarter-hour of day, per season and weekday/weekend", fontsize=16)
plt.tight_layout()
plt.show()

**This is the strongest calendar structure in the whole notebook, and it is strongly seasonal.**
The daily shape is nearly flat in winter — a median amplitude of about 44 EUR/MWh — and more than
three times as large in spring and summer, at roughly 148. Autumn sits between them. So there is
a daily cycle, but only for part of the year, which is one reason the single pooled decomposition
of §4.2 finds so little.

**In spring and summer the midday median falls essentially to zero.** Both seasons trough at
**13:45** — spring at −0.1 and summer at +5.4 EUR/MWh — and peak in the evening around 20:30–21:00.
That is the solar signature: at the solar maximum the system is routinely long and the price of
being short collapses; as solar ramps down into the evening peak, balancing gets expensive. Winter
has no such trough, and its shallow maximum sits in the afternoon instead.

**Again the effect is far larger in the tails than in the level.** Within summer, the share of
quarter-hours below zero reaches **47 % at 13:45** and falls to **2 % at 22:00**; in winter the
same curve peaks at 22 % in the early morning. A median that moves by 148 EUR/MWh across the day
corresponds to a negative-price probability that moves by more than twenty-fold.

**The weekday/weekend split behaves as §3.4 predicted**, and differently from the team's load
case: rather than a near-constant vertical offset with the same shape, the weekend curve deepens
the midday trough specifically — the two effects compound where they meet, at midday in summer.

### 3.6 — Federal holidays

**Skipped.** Team §3.6 compares grid load on federal holidays with same-weekday neighbours; the
mechanism (offices and factories closed) acts on demand, not directly on the imbalance price.
Left out of scope for this exploration rather than ported without a hypothesis.

### 3.7 — Where the price tails sit on the calendar

The extreme 1 % of quarter-hours at each end, selected **by rank**: a descriptive slice, not a
threshold, and no flag column is written back onto `time_series`. Blue for the low (most
negative) end, red for the high end — the same pairing the team uses for the residual-load tails.

In [ ]:
n_tail = round(0.01 * len(time_series))
low_tail = price.nsmallest(n_tail)   # descriptive slice, never persisted
high_tail = price.nlargest(n_tail)
TAIL_COLOR = {"low": "#2C6EBA", "high": "#B10F0F"}

print(f"{n_tail:,} quarter-hours in each tail (1 % of the record by rank)")
print(f"  low  tail spans {low_tail.min():>10,.2f} .. {low_tail.max():>10,.2f} {UNIT}")
print(f"  high tail spans {high_tail.min():>10,.2f} .. {high_tail.max():>10,.2f} {UNIT}")

DIMENSIONS = [
    ("year", lambda idx: idx.year, YEARS),
    ("month", lambda idx: idx.month, range(1, 13)),
    ("hour of day", lambda idx: idx.hour, range(24)),
    ("day of week", lambda idx: idx.dayofweek, range(7)),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

for ax, (label, extract, full_range) in zip(axes.flat, DIMENSIONS):
    frame = pd.DataFrame(
        {
            "lowest 1 %": pd.Series(extract(low_tail.index)).value_counts(normalize=True),
            "highest 1 %": pd.Series(extract(high_tail.index)).value_counts(normalize=True),
        }
    ).reindex(list(full_range)).fillna(0) * 100

    x = np.arange(len(frame))
    ax.bar(x - 0.2, frame.iloc[:, 0].to_numpy(), width=0.4, label=frame.columns[0], color=TAIL_COLOR["low"])
    ax.bar(x + 0.2, frame.iloc[:, 1].to_numpy(), width=0.4, label=frame.columns[1], color=TAIL_COLOR["high"])
    ax.set_xticks(x)
    ax.set_xticklabels(DAY_NAMES if label == "day of week" else [str(v) for v in frame.index], fontsize=9)
    ax.set_title(f"by {label}", fontsize=12, pad=8)
    ax.set_ylabel("share of that tail (%)", color="grey")
    ax.set_xlabel(label)
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False, fontsize=9)

fig.suptitle("Where the two reBAP tails sit on the calendar (1 % by rank each)", fontsize=16)
plt.tight_layout()
plt.show()

**The two tails are mirror images by hour and by day type, exactly as the team's residual-load
tails are — but they trend the same way as each other, not in opposite directions.**

**By hour and weekday the mechanism story holds.** The low tail concentrates at **11:00–14:00**
(about 47 % of it in those four hours) and is **over-represented at weekends** (38 % against a
28.6 % baseline). The high tail is the mirror: **18:00–21:00** and heavily **weekday** (14 %
weekend, half the baseline). Midday-and-weekend for oversupply, evening-and-weekday for tight
margins — the same two mechanisms §3.5 found in the medians, seen in the extremes.

**By year both tails shrink, and this is where reBAP parts company with residual load.** The team
reports a low tail that is "growing and seasonal" against a high tail that is "stable and
structural". Here **both** are concentrated in the early record: 47 % of the low tail and 70 % of
the high tail fall in 2022 alone, declining to 4–5 % each in 2026. Extreme prices in both
directions have become markedly rarer. §4.4 shows the same compression in the ramps.

**By month the two differ as expected.** The low tail is a spring phenomenon — April alone holds
20 % of it, with March and May next — which is the high-solar, low-demand combination. The high
tail is overwhelmingly **August (38 %)** and September, which is 2022's crisis showing through
rather than a stable seasonal pattern; with 70 % of that tail sitting in one year, its month
profile is largely that year's profile.

### 3.8 — Two representative extreme episodes

Selected by a **stated, reproducible, rank-based rule**, mirroring the team's two rules:

1. **The longest run of consecutive negative prices.** Ties broken by earliest start.
2. **The highest 24-hour rolling mean price.** Time-based window (`rolling("24h")`) on the
   tz-aware index, `min_periods=STEPS_PER_DAY` so a partial window at the start cannot win.

In [ ]:
negative = price < 0

# --- rule 1: longest consecutive negative run, ties broken by earliest start
blocks = (negative != negative.shift()).cumsum()
neg_runs = (
    pd.DataFrame(
        {
            "length": negative.groupby(blocks).size(),
            "is_negative": negative.groupby(blocks).first(),
            "start": time_series.index.to_series().groupby(blocks).min(),
        }
    )
    .query("is_negative")
    .sort_values(["length", "start"], ascending=[False, True])
)
neg_runs["hours"] = neg_runs["length"] / STEPS_PER_HOUR
print("longest negative runs (top 5):")
display(neg_runs.head(5)[["start", "length", "hours"]].reset_index(drop=True))

run_start = neg_runs.iloc[0]["start"]
run_end = run_start + RESOLUTION * (int(neg_runs.iloc[0]["length"]) - 1)

# --- rule 2: highest 24-hour rolling mean, on a TIME-based window
rolling_24h = price.rolling("24h", min_periods=STEPS_PER_DAY).mean()
peak_end = rolling_24h.idxmax()
peak_start = peak_end - pd.Timedelta("24h") + RESOLUTION
print(f"\nhighest 24 h rolling mean: {rolling_24h.max():,.1f} {UNIT} over {peak_start} .. {peak_end}")

In [ ]:
episodes = [
    (f"Longest negative run: {int(neg_runs.iloc[0]['length'])} consecutive quarter-hours "
     f"({neg_runs.iloc[0]['hours']:.2f} h)", run_start, run_end),
    (f"Highest 24 h rolling mean: {rolling_24h.max():,.0f} {UNIT}", peak_start, peak_end),
]

for title, start, end in episodes:
    window = price.loc[start - pd.Timedelta("36h"): end + pd.Timedelta("36h")]
    x = wall_clock(window.index)

    fig, ax = plt.subplots(figsize=(15, 5.5))
    ax.plot(x, window.to_numpy(), color=PRICE_COLOR, lw=1.4, label=PRICE)
    ax.axhline(0, color="0.35", linewidth=0.9)
    ax.axvspan(wall_clock(pd.DatetimeIndex([start]))[0], wall_clock(pd.DatetimeIndex([end]))[0],
               color="0.88", zorder=0, label="selected episode")

    style_timeseries(
        ax,
        f"{title}\n{start:%Y-%m-%d %H:%M} to {end:%Y-%m-%d %H:%M} local (+/- 36 h context)",
        UNIT,
    )
    # style_timeseries assumes a multi-year axis; a few-day window needs day ticks.
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=(0, 6, 12, 18)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
    ax.set_xlabel("Europe/Berlin local time")
    ax.legend(frameon=False, fontsize=9)
    plt.tight_layout()
    plt.show()

**Negative prices arrive in hours, and they never last a day.** The longest unbroken run of
negative reBAP in nearly five years is **42 quarter-hours — ten and a half hours** — starting on a
Monday morning in April 2026, and the runners-up (40, 35, 35, 34 steps) are barely shorter. Every
one of them is a daytime run, which is §3.5's midday trough in its most extreme form. There is no
multi-day negative episode anywhere in the record.

That matters for how this series should be thought about. The team's residual load has a
*Dunkelflaute* mode that persists for days; the imbalance price has no equivalent. Even its most
extreme sustained state resolves within a single solar cycle, which is consistent with §4.3's
finding that the price changes sign between consecutive quarter-hours about a tenth of the time.

**The expensive episode is a single day, and a famous one.** The highest 24-hour rolling mean
(about 1,228 EUR/MWh) ends on the evening of **3 June 2024** — the same episode that supplied five
of the ten highest quarter-hours in §2.2 and the record maximum of 14,999.99. The ±36 h context
shows the surrounding days at ordinary levels: this is one day, not a regime.

The contrast in duration between the two episodes is the point. Oversupply and scarcity both
arrive as **events of hours**, not states of days — the opposite of the asymmetry the team found
in residual load, where oversupply arrives in hours and tight margins in days.

### 3.9 — Self-check

In [ ]:
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert len(time_series) == LOADED["rows"]
assert len(low_tail) == len(high_tail) == n_tail
assert low_tail.max() < high_tail.min(), "the two tails overlap"
assert run_end > run_start and peak_end > peak_start
assert (price.loc[run_start:run_end] < 0).all(), "selected run contains a non-negative price"

print("univariate/time-structure self-check passed")
print(f"  {n_tail:,} quarter-hours per tail, ranges disjoint")
print(f"  no column added to time_series ({len(time_series.columns)} columns), {len(time_series):,} rows")

---

## 4 — Temporal dependence, decomposition and ramps

Mirrors team §6.1–6.2 and §6.4–6.5. Team §6.3 merged a distribution panel with a ramp panel;
the distribution half already ran as §3.2 here, so §4.3 below carries the ramp half alone.

One structural advantage over the team notebook, established in §2.1: this record is **gap-free
in absolute time**. An ACF counts lags by position, so in the SMARD data every lag after a spring
DST gap is off by an hour; here one lag step is always exactly fifteen minutes. The gap-handling
machinery is kept anyway — it costs nothing and it is what makes these cells survive a re-fetch
that does have gaps — but it is a no-op, and the cells assert that rather than assume it.

### 4.1 — Autocorrelation

Two figures. The first is the raw ACF/PACF grid — one week of lags (672 steps), then zooms on the
first two days and on the second-to-third day. The second removes the **weekday × quarter-hour**
mean first and asks what dependence is left once the calendar structure of §3.4 and §3.5 is taken
out.

Lag markers sit at 96 (one day) and 672 (one week), the 15-minute analogues of the team's 24 and
168.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf

LAG_DAY, LAG_WEEK = STEPS_PER_DAY, STEPS_PER_WEEK


def longest_gapfree_segment(col):
    """Longest run of consecutive rows with no missing value in `col`.

    Breaks on both a missing value and an index gap: `spans_gap` marks the row after each jump,
    and a lag count that runs across one of those is off from there on. Copied in spirit from
    team-EDA.ipynb §6.1; here it returns the whole record, which §4.6 asserts.
    """
    usable = time_series[col].notna() & ~time_series["spans_gap"]
    blocks = (~usable).cumsum()
    lengths = usable.groupby(blocks).sum()
    best = lengths.idxmax()
    return time_series.loc[(blocks == best) & usable, col]


segment = longest_gapfree_segment(PRICE)
print(
    f"{PRICE} longest gap-free segment: {len(segment):,} of {len(time_series):,} rows "
    f"({100 * len(segment) / len(time_series):.1f} %)"
)
print(f"  {wall_clock(segment.index).min():%Y-%m-%d %H:%M} .. {wall_clock(segment.index).max():%Y-%m-%d %H:%M}")

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 9))
values = price.to_numpy()

plot_acf(values, lags=LAG_WEEK, ax=axes[0, 0])
plot_pacf(values, lags=LAG_WEEK, method="ywm", ax=axes[0, 1])
plot_acf(values, lags=2 * LAG_DAY, ax=axes[1, 0])
plot_pacf(values, lags=2 * LAG_DAY, method="ywm", ax=axes[1, 1])
plot_acf(values, lags=range(2 * LAG_DAY, 3 * LAG_DAY), ax=axes[2, 0])
plot_pacf(values, lags=range(2 * LAG_DAY, 3 * LAG_DAY), method="ywm", ax=axes[2, 1])

# set_title AFTER the plot calls: statsmodels writes its own title into the axes.
for ax, title in zip(
    axes.flat,
    [f"ACF — reBAP — {LAG_WEEK} lags (1 week)", f"PACF — reBAP — {LAG_WEEK} lags (ywm)",
     f"ACF — zoom 0-{2 * LAG_DAY} (2 days)", f"PACF — zoom 0-{2 * LAG_DAY} (ywm)",
     f"ACF — zoom {2 * LAG_DAY}-{3 * LAG_DAY} (day 3)", f"PACF — zoom {2 * LAG_DAY}-{3 * LAG_DAY} (ywm)"],
):
    ax.set_title(title, fontsize=12, pad=8)
    ax.set_xlabel("lag (15-minute steps)")
    ax.set_ylabel("correlation")

for ax in axes[0]:
    for lag in (LAG_DAY, LAG_WEEK):
        ax.axvline(lag, color="#B10F0F", linestyle=":", linewidth=1)

plt.tight_layout()
plt.show()

raw_acf = acf(values, nlags=LAG_WEEK, fft=True)
print("raw ACF at the lags that matter:")
for lag, name in [(1, "15 min"), (4, "1 hour"), (LAG_DAY, "1 day"), (2 * LAG_DAY, "2 days"), (LAG_WEEK, "1 week")]:
    print(f"  lag {lag:>3} ({name:<6}) : {raw_acf[lag]:.3f}")

In [ ]:
# What is left once the calendar mean is removed? Weekday x quarter-hour = 7 x 96 = 672 cells.
qh_of_day = segment.index.hour * STEPS_PER_HOUR + segment.index.minute // 15
calendar_mean = segment.groupby([segment.index.dayofweek, qh_of_day]).transform("mean")
adjusted = segment - calendar_mean

adj_acf = acf(adjusted.to_numpy(), nlags=LAG_WEEK, fft=True)
confidence = 1.96 / np.sqrt(len(adjusted))
lags = np.arange(len(adj_acf))

fig, ax = plt.subplots(figsize=(13.8, 4.6), constrained_layout=True)
ax.axhspan(-confidence, confidence, color="#E3E8EF", zorder=0)
ax.vlines(lags, 0, adj_acf, color=PRICE_COLOR, lw=0.9, alpha=0.85)
ax.axhline(0, color="#AEB8C5", lw=1.0)
for lag in (LAG_DAY, LAG_WEEK):
    ax.axvline(lag, color=MUTED, ls="--", lw=1.0, alpha=0.8)

ax.set_title(
    f"reBAP — ACF after removing weekday × quarter-hour means ({len(adjusted):,} steps)",
    fontsize=13, pad=8,
)
ax.set_xlabel("lag (15-minute steps)")
ax.set_ylabel("correlation")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.show()

print(f"calendar cells: 7 weekdays x {STEPS_PER_DAY} quarter-hours = {7 * STEPS_PER_DAY}, "
      f"~{len(segment) // (7 * STEPS_PER_DAY):,} observations each")
print(f"variance removed by the calendar mean: {100 * (1 - adjusted.var() / segment.var()):.1f} %")
print("de-seasonalized ACF:")
for lag, name in [(1, "15 min"), (4, "1 hour"), (LAG_DAY, "1 day"), (LAG_WEEK, "1 week")]:
    print(f"  lag {lag:>3} ({name:<6}) : {adj_acf[lag]:.3f}")

**reBAP is far closer to noise than load is, and it is very nearly calendar-free.** Three
readings, in increasing order of how much they matter.

**The raw ACF decays fast and has no strong calendar peaks.** Correlation is 0.58 one step back,
0.35 an hour back, and only **0.15 at one day** and **0.11 at one week**. Compare the team's
§6.1, where both load series stay around 0.7–0.9 out to a full week with pronounced peaks at
every daily multiple. The daily and weekly echoes exist here — they are visible as gentle ripples
— but they are an order of magnitude weaker than the ones that dominate load.

**Removing the calendar changes almost nothing.** The weekday × quarter-hour mean — 672 cells,
~244 observations each — accounts for just **2.6 %** of the variance, and the de-seasonalized ACF
is virtually identical to the raw one (0.575 vs 0.581 at one step, 0.133 vs 0.147 at one day).
This is the single biggest structural difference from the team's series: for grid load, knowing
the weekday and the hour tells you most of what you need; for reBAP it tells you almost nothing.

**What is left is short memory, not structure.** The PACF concentrates in the first few lags and
then dies. Practically: a lagged reBAP feature is worth something at the 15-minute to one-hour
horizon and very little beyond it, and a "same quarter-hour yesterday" feature — the backbone of
a load model — should not be expected to carry this series.

### 4.2 — Seasonal decomposition

STL with a **96-step (one-day) period**, `robust=True`, over the full record — the 15-minute
analogue of the team's `period=24`. Because the record is gap-free, the approximation the team
had to document (a 23-hour spring day breaking `period=24`) does not arise here: every day in
this decomposition really does have 96 observations, except the two DST days, which have 92 and
100 in local terms but a constant 96 in the absolute time the index is built on.

`robust=True` is kept for faithfulness, and it matters more here than for load: without it a
handful of four-figure spikes would drag the trend around.

In [ ]:
from statsmodels.tsa.seasonal import STL
from time import perf_counter

started = perf_counter()
# Values, not the Series: STL wants a regular array, and the index carries a timezone.
stl_result = STL(price.to_numpy(), period=STEPS_PER_DAY, robust=True).fit()
print(f"STL fitted in {perf_counter() - started:.1f} s (period={STEPS_PER_DAY}, robust=True)")

In [ ]:
x_all = wall_clock(price.index)
components = [
    ("Observed", price.to_numpy(), PRICE_COLOR),
    ("Trend", stl_result.trend, PRICE_COLOR),
    ("Daily seasonal", stl_result.seasonal, "#D9A53A"),
    ("Remainder", stl_result.resid, MUTED),
]

fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)
for ax, (name, vals, color) in zip(axes, components):
    ax.plot(x_all, vals, color=color, linewidth=0.5)
    ax.set_ylabel(f"{name}\n({UNIT})", color="grey")
    ax.grid(axis="y", color="0.92", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axes[-1].axhline(0, color="#1C1C1C", linewidth=0.8)
axes[-1].set_xlabel("")

fig.suptitle("STL decomposition of reBAP (one-day period, 96 steps)", fontsize=15)
plt.tight_layout()
plt.show()

strength = 1 - np.var(stl_result.resid) / np.var(stl_result.seasonal + stl_result.resid)
print(f"daily-seasonal strength : {strength:.2f}")
print(f"remainder std           : {np.std(stl_result.resid):,.1f} {UNIT}")
print(f"trend range             : {stl_result.trend.min():,.1f} .. {stl_result.trend.max():,.1f} {UNIT}")
print(f"seasonal component range: {stl_result.seasonal.min():,.1f} .. {stl_result.seasonal.max():,.1f} {UNIT}")

The printed `seasonal component range` above is **wider than almost all of the price data**, which
is worth taking seriously rather than reporting as a strength figure. STL's seasonal component is
*local*: with the default smoother each quarter-hour-of-day sub-series is smoothed over only a few
neighbouring days, so it can track a one-off event instead of describing a repeating daily shape.
The cell below separates the two possibilities.

In [ ]:
seasonal = pd.Series(stl_result.seasonal, index=price.index)
qh_index = seasonal.index.hour * STEPS_PER_HOUR + seasonal.index.minute // 15

within = seasonal.groupby(qh_index).std().mean()      # how much each cell wanders over time
between = seasonal.groupby(qh_index).mean().std()      # how much the average shape varies by time of day
average_shape = seasonal.groupby(qh_index).mean()

print(f"seasonal std overall                    : {seasonal.std():,.1f} {UNIT}")
print(f"|seasonal| > 1000                       : {int((seasonal.abs() > 1000).sum())} of "
      f"{len(seasonal):,} steps ({100 * (seasonal.abs() > 1000).mean():.2f} %)")
print(f"mean WITHIN-cell std (wander over time) : {within:,.1f} {UNIT}")
print(f"BETWEEN-cell std (the average shape)    : {between:,.1f} {UNIT}")
print(f"ratio within/between                    : {within / between:.1f}x")
print(f"average daily shape amplitude           : {average_shape.max() - average_shape.min():,.1f} {UNIT} "
      f"(peak {average_shape.idxmax() / STEPS_PER_HOUR:.2f} h, trough {average_shape.idxmin() / STEPS_PER_HOUR:.2f} h)")

fig, ax = plt.subplots(figsize=(12, 4.2))
ax.plot(average_shape.index / STEPS_PER_HOUR, average_shape.to_numpy(), color="#D9A53A", lw=2)
ax.axhline(0, color=MUTED, lw=0.8)
ax.set_title("Average of the STL daily-seasonal component by quarter-hour of day", fontsize=13, pad=8)
ax.set_xlabel("hour of day (Europe/Berlin)")
ax.set_ylabel(UNIT, color="grey")
ax.set_xticks(range(0, 25, 3))
ax.grid(axis="y", color="0.92", lw=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

**The extreme seasonal values are rare, but the component is genuinely unstable.** Only about
0.07 % of steps have a seasonal value beyond ±1000, so the printed range is driven by a handful
of cases — `robust=True` did catch most of the big spikes, leaving them in the remainder where
they belong. The real issue is the ratio: the seasonal component **wanders over time roughly five
times more than the average daily shape varies across the day**. Averaged over the whole record
that underlying shape is modest — an amplitude of roughly 120 EUR/MWh peak-to-trough, with an
evening high around 18:45 and an early-afternoon low around 13:45 — and it is dwarfed by a
remainder whose standard deviation is around 190 EUR/MWh.

That shape is at least physically sensible: the early-afternoon low coincides with the solar
maximum and the evening high with the solar ramp-down, which is when balancing is hardest. But
it is a weak effect, and the notebook does not test that reading — relating price to generation
would need the SMARD data this notebook deliberately does not load.

So the reported **daily-seasonal strength of 0.35 overstates the case**: it credits the seasonal
component for variation that is really local wander, not a repeating cycle. Read together with
§4.1's 2.6 % calendar variance, the two methods agree — there is a weak daily shape in reBAP, and
it explains very little.

A stable-shape alternative exists (a much longer `seasonal=` smoother, or simply the calendar
means of §4.1) and would be the right tool if a daily profile were the goal. It is not applied
here: the decomposition is kept in the team's configuration so the comparison is like for like,
and the caveat is recorded instead of tuned away.

### 4.3 — Ramps

The ramp is the change from one quarter-hour to the next, in **EUR/MWh per 15 min**. For a price
this is not an engineering constraint the way a load ramp is — nothing has to physically follow
it — but it measures how much of the price is news rather than state, which is exactly what
decides whether a lagged price is a useful feature.

The mask on `spans_gap` is retained from the team's cell so a re-fetch with gaps stays correct.
Here it removes nothing, and §4.6 asserts that the only `NaN` is the first row.

In [ ]:
# ramp stays LOCAL to this section — never written back onto time_series.
ramp = price.diff().mask(time_series["spans_gap"])
abs_ramp = ramp.abs().dropna()

RAMP_QUANTILES = [0.50, 0.80, 0.90, 0.95, 0.975, 0.99, 0.995]
ramp_percentiles = abs_ramp.quantile(RAMP_QUANTILES)
RAMP_COLOR = "#E76F51"

shape = pd.Series(
    {
        "mean |ramp|": abs_ramp.mean(),
        "median |ramp|": abs_ramp.median(),
        "std of ramp": ramp.std(),
        "max |ramp|": abs_ramp.max(),
        "ramp skew": ramp.skew(),
        "ramp excess kurtosis": ramp.kurt(),
        "|ramp| > 100": int((abs_ramp > 100).sum()),
        "|ramp| > 100, share %": 100 * (abs_ramp > 100).mean(),
        "sign changes, share %": 100 * (np.sign(price) != np.sign(price.shift())).mean(),
    }
)
display(shape.round(2).to_frame(f"reBAP ramp ({UNIT} per 15 min)"))

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.4), gridspec_kw={"width_ratios": [1.15, 1]},
                         constrained_layout=True)

lo_r, hi_r = np.percentile(ramp.dropna(), [1, 99])
body = ramp.dropna()
body = body[(body >= lo_r) & (body <= hi_r)]
axes[0].hist(body, bins=160, color=PRICE_COLOR)
axes[0].axvline(0, color="#B10F0F", linewidth=1.4)
axes[0].set_title(f"Ramp distribution, central 99 % ({lo_r:,.0f} to {hi_r:,.0f})", fontsize=13, pad=8)
axes[0].set_xlabel(f"{UNIT} per 15 min")
axes[0].set_ylabel("quarter-hours", color="grey")

axes[1].plot(np.array(RAMP_QUANTILES) * 100, ramp_percentiles.to_numpy(),
             color=RAMP_COLOR, marker="o", markersize=5.5, lw=2.5)
for q in (0.90, 0.99):
    axes[1].annotate(f"P{q * 100:g}  {ramp_percentiles.loc[q]:,.0f}",
                     xy=(q * 100, ramp_percentiles.loc[q]), xytext=(8, -4),
                     textcoords="offset points", fontsize=10, color=MUTED)
axes[1].margins(x=0.10, y=0.12)
axes[1].set_title("Upper tail of 15-minute reBAP ramps", fontsize=13, pad=8)
axes[1].set_xlabel("ramp percentile")
axes[1].set_ylabel(f"|15-min ramp| ({UNIT})", color="grey")

for ax in axes:
    ax.grid(axis="y", color="0.92", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
plt.show()

print(f"ramp NaNs: {int(ramp.isna().sum())} "
      f"(expected {int(time_series['spans_gap'].sum()) + 1}: one per gap plus the first row)")

**A typical quarter-hour barely moves; a rare one moves enormously.** The median absolute step is
about 21 EUR/MWh against a mean of about 70 — the mean is more than three times the median, which
only happens when a minority of steps dominate. The excess kurtosis of the ramp distribution is in
the hundreds, and the largest single step exceeds 13,000 EUR/MWh.

Two further numbers are worth carrying forward. About **17 % of all steps move by more than
100 EUR/MWh**, which is roughly the median price level — so in one step in six, the price changes
by about as much as its own typical value. And the price **changes sign between consecutive steps
about 10 % of the time**: the negative hours of §3.2 are not long calm stretches but are
interleaved with positive ones at quarter-hour granularity.

Together with §4.1 this settles the character of the series for modelling purposes: reBAP is
dominated by innovations, not by state. That is a statement about the price, not about the
project's target — nothing here bears on residual load, which is modelled elsewhere.

### 4.4 — Ramps by year

The same quantiles split by year, on a **matched calendar window** — 1 January to the record's
last calendar date, every year — so the partial final year is compared like for like. The team
inherits this window from its §5; §5 is out of scope here, so the window is built locally.

In [ ]:
# Matched window: 1 Jan .. the record's own last calendar date, in every year.
wall = wall_clock(time_series.index)
last_day = wall.max()
in_window = (wall.month < last_day.month) | (
    (wall.month == last_day.month) & (wall.day <= last_day.day)
)
matched = time_series.loc[in_window]
matched_wall = wall_clock(matched.index)
partial_years = {int(last_day.year)}

print(f"matched window: 1 Jan .. {last_day:%d %b}, {len(matched):,} of {len(time_series):,} rows")
print(f"partial year(s): {sorted(partial_years)}")

RAMP_YEAR_QUANTILES = [0.50, 0.90, 0.95, 0.99]
matched_ramp = ramp.abs().loc[matched.index].dropna()
ramp_by_year = (
    matched_ramp.groupby(wall_clock(matched_ramp.index).year)
    .quantile(RAMP_YEAR_QUANTILES)
    .unstack()
)
ramp_by_year.columns = [f"P{q * 100:g}" for q in RAMP_YEAR_QUANTILES]
display(ramp_by_year.round(1))

quantile_shades = plt.get_cmap("YlOrRd")(np.linspace(0.35, 0.92, len(ramp_by_year.columns)))
MARKERS = ["o", "o", "D", "s"]

fig, ax = plt.subplots(figsize=(13, 5.8))
for marker, column, shade in zip(MARKERS, ramp_by_year.columns, quantile_shades):
    ax.plot(ramp_by_year.index, ramp_by_year[column].to_numpy(),
            color=shade, marker=marker, lw=2.2, label=column)

ax.set_title("15-minute reBAP ramps by year (matched calendar window)", fontsize=15, pad=12)
ax.set_xlabel(f"year (matched 1 Jan – {last_day:%d %b} window)")
ax.set_ylabel(f"|15-min ramp| ({UNIT})", color="grey")
ax.set_xticks(list(ramp_by_year.index))
ax.grid(axis="y", color="0.92", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
ax.legend(ncol=len(ramp_by_year.columns), frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

growth = (ramp_by_year.iloc[-1] / ramp_by_year.iloc[0] - 1) * 100
print(f"change from {ramp_by_year.index[0]} to {ramp_by_year.index[-1]} (%):")
print(growth.round(1).to_string())

**This is the opposite of what the team found for residual load, and it is the clearest trend in
the notebook.** Team §6.4 reports residual-load ramps *growing*, with the upper percentiles
growing fastest — the gap between a typical hour and an extreme one widening year on year. reBAP
ramps do the reverse: **every percentile falls**, and the upper ones fall hardest. P90 drops by
about 80 % across the record while the median drops by about 65 %, so the distribution is not just
shifting down, it is **compressing**.

The two findings are not in conflict — they describe different things. Residual load is becoming
more volatile in physical terms; the price of being imbalanced has become less volatile over the
same period. Whatever the cause (the 2022 energy-crisis level unwinding, a deeper balancing
market, regulatory change), this notebook can only record the pattern, not attribute it: nothing
in this dataset identifies a mechanism.

One caution on reading the 2022 column: §3.1 showed 2022 was an exceptional price year at every
aggregation, so "the trend" here is substantially "2022 was extreme and the years since have not
been". The 2023 → 2026 decline is real but far gentler than the 2022 → 2026 headline.

### 4.5 — Price level by year

Median with the P10–P90 range, on the same matched window. For a series this spiky the P10–P90
band says far more than the mean, which §3.1's per-year table showed is dragged by a handful of
quarter-hours.

In [ ]:
matched_price = matched[PRICE]
annual_level = matched_price.groupby(matched_wall.year).agg(
    median="median",
    p10=lambda s: s.quantile(0.10),
    p90=lambda s: s.quantile(0.90),
    neg_pct=lambda s: 100 * (s < 0).mean(),
    n="count",
)
display(annual_level.round(1))

level_shades = plt.get_cmap("viridis")(np.linspace(0.05, 0.95, len(annual_level)))

fig, ax = plt.subplots(figsize=(11.8, 6.1), constrained_layout=True)
for (year, row), color in zip(annual_level.iterrows(), level_shades):
    ax.plot([year, year], [row["p10"], row["p90"]], lw=6, color=color, alpha=0.30,
            solid_capstyle="round", zorder=1)
    ax.scatter(year, row["median"], s=110, color=color, edgecolor="white", linewidth=0.8, zorder=3)
    ax.text(year, row["p90"] + 0.04 * annual_level["p90"].max(), f"n={int(row['n']):,}",
            va="bottom", ha="center", fontsize=9.5, color=MUTED)

ax.axhline(0, color="#B10F0F", lw=1.2)
ax.set_xticks(
    list(annual_level.index),
    [f"{y}\n· partial" if y in partial_years else str(y) for y in annual_level.index],
)
ax.set_title("reBAP level by year — matched calendar window", fontsize=15, pad=12)
ax.set_xlabel("year")
ax.set_ylabel(f"{UNIT}, median with P10–P90 range", color="grey")
ax.grid(axis="y", color="0.92", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
plt.show()

**The level falls and then stabilises; the negative share peaks in the middle of the record, not
at the end.** Median price drops sharply after 2022 and then sits in a narrow band, with the last
(partial) year slightly higher than the two before it. The P10–P90 band narrows in the same way
the ramps did — the compression of §4.4 is visible in levels too.

The negative share is the one non-monotonic column, and it is worth stating plainly because the
intuitive story would be wrong. On the matched window it rises from about 14 % (2022) to a peak
of about 23 % in **2024**, then falls back to about 15 % and 11 %. So negative imbalance prices
did *not* simply become more common as renewable capacity grew — on this record they peaked
mid-way and have receded since.

Two cautions before anyone builds on that. The final year is **partial** (1 Jan – 6 Sep), and
§3.7 showed the tails have a seasonal signature, so a window ending in early September is not
neutral with respect to them — the matched window makes the years comparable to each other, not
to a full calendar year. And the P10 recovery towards zero in the last two years is measured on
the same partial window. These are observations about five annual windows, not a demonstrated
reversal.

### 4.6 — Self-check

In [ ]:
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert "ramp" not in time_series.columns, "ramp must stay local to this section"
assert int(ramp.isna().sum()) == int(time_series["spans_gap"].sum()) + 1, int(ramp.isna().sum())
# The record is gap-free (§2.1), so the "longest gap-free segment" must be the whole record.
assert len(segment) == len(time_series), (len(segment), len(time_series))
step = segment.index.to_series().diff().dropna()
assert (step == RESOLUTION).all(), "segment is not gap-free"
assert len(stl_result.trend) == len(price)
assert list(ramp_by_year.index) == sorted(ramp_by_year.index)
assert set(annual_level.index) <= set(YEARS)
assert len(time_series) == LOADED["rows"]

print("temporal-dependence self-check passed")
print(f"  ramp NaNs == spans_gap + 1 == {int(ramp.isna().sum())}")
print(f"  gap-free segment == full record ({len(segment):,} steps of {RESOLUTION})")
print(f"  STL fitted over the full record, period {STEPS_PER_DAY}")
print(f"  no column added to time_series ({len(time_series.columns)} columns)")

---

## 5 — Context appendix, findings and closing self-check

Mirrors team §7. Two of its four appendix items carry over: **completeness by calendar month**
(§5.1) and the **duration curve** (§5.2). Team §7.2 ("which system variables move together") and
§7.4 ("wind and solar support through the year") have no counterpart in a single-series,
standalone notebook and are not ported.

### 5.1 — Completeness by calendar month

§2.1 established that the record is gap-free overall. This breaks that down by month, so a
re-fetch that loses part of one month shows up here rather than being averaged away. The expected
count per month is computed in **absolute time** from the tz-aware month boundaries, so March and
October automatically expect 4 fewer and 4 more quarter-hours than a naive `days × 96`.

In [ ]:
months = wall_clock(time_series.index).to_period("M")
observed = time_series.groupby(months).size()

# Expected = the real duration of each local calendar month, divided by the resolution. Localizing
# the boundaries is what makes the DST months come out right without a special case.
starts = pd.DatetimeIndex(observed.index.start_time).tz_localize("Europe/Berlin")
ends = pd.DatetimeIndex((observed.index + 1).start_time).tz_localize("Europe/Berlin")
expected = ((ends - starts) / RESOLUTION).astype(int)

completeness = pd.DataFrame(
    {"observed": observed.to_numpy(), "expected": expected},
    index=observed.index,
)
completeness["missing"] = completeness["expected"] - completeness["observed"]
completeness["complete_%"] = 100 * completeness["observed"] / completeness["expected"]

naive = completeness.index.days_in_month * STEPS_PER_DAY
dst_months = completeness[completeness["expected"] != naive]
print("months where the real duration differs from days x 96 (the DST months):")
display(dst_months.assign(naive_expected=naive[completeness["expected"] != naive]))

incomplete = completeness[completeness["complete_%"] < 100]
print(f"\nincomplete months: {len(incomplete)} of {len(completeness)}")
display(incomplete if len(incomplete) else completeness.tail(3))

grid = pd.Series(
    completeness["complete_%"].to_numpy(),
    index=pd.MultiIndex.from_arrays(
        [completeness.index.year, completeness.index.month], names=["year", "month"]
    ),
).unstack()

fig, ax = plt.subplots(figsize=(13, 3.4))
sns.heatmap(
    grid, annot=True, fmt=".0f", cmap="YlGnBu", vmin=0, vmax=100, ax=ax,
    cbar_kws={"label": "% of expected quarter-hours present"}, annot_kws={"fontsize": 8},
    linewidths=0.5, linecolor="white",
)
ax.set_title("Data completeness by calendar month (100 = every quarter-hour present)", fontsize=13, pad=10)
ax.set_xlabel("month")
ax.set_ylabel("year")
plt.tight_layout()
plt.show()

**Every complete month is 100 % complete, and the only partial month is the last one.** The
heatmap is uniform except September 2026, where the record simply ends on the 6th. The DST table
above is the check that the expected counts are right rather than merely matching: the five March
months expect 4 quarter-hours *fewer* than `days × 96` and the four October months expect 4
*more*, and the observed counts hit those numbers exactly.

That is a stronger completeness statement than the SMARD data supports, and it is the second time
this notebook has had to record a difference in the source's favour (§2.1 was the first).

### 5.2 — Duration curve

The price analogue of the team's load duration curve: every quarter-hour sorted from highest to
lowest, plotted against the share of time at or above that level. It answers "how often is the
price at least X", which the time-ordered views of §3.1 cannot.

Two versions. The first is the whole record on a symlog axis, for the same reason §3.2 needed
one. The second is **one curve per year on a linear axis restricted to the central range**, which
is where the year-on-year compression found in §4.4 and §4.5 becomes visible as a shape rather
than a table.

In [ ]:
sorted_all = np.sort(price.to_numpy())[::-1]
share = np.linspace(0, 100, len(sorted_all))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.4), constrained_layout=True)

axes[0].plot(share, sorted_all, color=PRICE_COLOR, lw=1.8)
axes[0].set_yscale("symlog", linthresh=10)
axes[0].axhline(0, color="#B10F0F", lw=1.0)
axes[0].set_title("Full record, symlog price axis", fontsize=13, pad=8)
axes[0].set_ylabel(f"{UNIT} (symlog)", color="grey")

year_shades = plt.get_cmap("viridis")(np.linspace(0.05, 0.95, len(YEARS)))
wall_years = wall_clock(price.index).year
for year, shade in zip(YEARS, year_shades):
    values_y = np.sort(price[wall_years == year].to_numpy())[::-1]
    axes[1].plot(np.linspace(0, 100, len(values_y)), values_y, color=shade, lw=1.8, label=str(year))
axes[1].set_ylim(-400, 900)
axes[1].axhline(0, color="#B10F0F", lw=1.0)
axes[1].set_title("By year, central range (note: 2026 is partial)", fontsize=13, pad=8)
axes[1].set_ylabel(UNIT, color="grey")
axes[1].legend(frameon=False, ncol=2, fontsize=9)

for ax in axes:
    ax.set_xlabel("% of quarter-hours at or above this price")
    ax.grid(axis="y", color="0.92", lw=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
plt.show()

crossings = pd.DataFrame(
    {
        f"% of time >= {level:,}": {
            year: 100 * (price[wall_years == year] >= level).mean() for year in YEARS
        }
        for level in (0, 100, 300, 1000)
    }
).round(1)
display(crossings)

**The curves are steep at both ends and flat in the middle**, which is the duration-curve
signature of a series that is usually unremarkable and occasionally extreme. The middle 80 % of
quarter-hours occupy a narrow band; the top and bottom few per cent run away vertically.

**Year on year the curves flatten**, and the table makes the compression concrete. The share of
time above 300 EUR/MWh collapses from **33.9 % in 2022 to 1.5–2.8 %** in every later year — more
than a tenfold drop, and by far the sharpest break in the table. Above 1,000 EUR/MWh the share
falls from 0.8 % to 0.2–0.3 %. The share **below zero** — the distance from the 100 % end up to
the red line — instead moves non-monotonically, from 16.3 % (2022) to a peak of 19.8 % in 2024 and
back to 10.9 %, the same shape §4.5 found on the matched window.

Note the two ends behave differently: the expensive end broke once, after 2022, and has stayed
low; the cheap end drifts up and down without a break. Whatever compressed the upper tail did not
act symmetrically.

Read with §4.4, the picture is consistent: the reBAP distribution has been **narrowing from both
ends** since 2022. Note again that 2026 is a partial year ending 6 September, so its curve omits
the autumn and winter months entirely.

---

## Findings

Recorded as **proposals from one member's exploration**, per the `EDA-<name>` convention — not as
project facts, and not as anything the team has reviewed. Every number below is produced by a cell
above. Nothing here defines a threshold, a flag or a risk case.

**On the data itself**

1. **The record is complete.** 164,156 quarter-hours, 2022-01-01 to 2026-09-06, zero missing
   values, zero duplicates, and **zero gaps in absolute time** (§2.1, §5.1). Both DST switches are
   handled by the source rather than lost, which is a genuine difference from SMARD — the
   "five spring gaps" intuition from the team notebook does **not** transfer.
2. **`unterdeckt` and `ueberdeckt` are identical on every row** (§1.2), asserted rather than
   assumed. The file carries one price twice.
3. **The index needs `bis_utc` to be unique.** 16 local timestamps repeat at the autumn folds;
   reconstructing the UTC offset resolves them with a zero-mismatch round trip (§1.3).
4. **No sign of a stuck feed.** The longest run of an unchanged price is 13 quarter-hours, and
   0.07 % of rows sit in runs longer than an hour (§2.2).

**On the series' behaviour**

5. **reBAP is very nearly calendar-free.** The weekday × quarter-hour mean explains **2.6 %** of
   the variance, and de-seasonalizing barely changes the ACF (§4.1). For grid load the equivalent
   structure is most of the signal.
6. **Autocorrelation is short.** 0.58 at one step, 0.15 at one day, 0.11 at one week (§4.1) — an
   order of magnitude weaker than the team's load series. A lagged price is useful at the
   15-minute-to-hour horizon and little beyond.
7. **What calendar structure exists is seasonal and lives in the tails.** The daily median
   amplitude is ~44 EUR/MWh in winter against ~148 in spring and summer, with both troughing at
   13:45; within summer the share of negative quarter-hours runs from **47 % at 13:45 to 2 % at
   22:00** (§3.5). Weekends raise the negative share by 72 % relative while moving the median only
   19 % (§3.4).
8. **Negative prices look like a separate state, not a tail.** On a symlog axis they form their
   own mode around −50 to −100 EUR/MWh (§3.2), unlike the team's residual load, which §6.3 there
   describes as a single continuum. 15.50 % of quarter-hours are negative.
9. **Everything extreme is brief.** The longest negative run in five years is 10.5 hours (§3.8),
   and the price changes sign between consecutive steps about 10 % of the time (§4.3). There is no
   multi-day analogue of a *Dunkelflaute* in this series.
10. **2022 is a regime, not a year.** It holds 47 % of the low tail and 70 % of the high tail
    (§3.7), its median August price is 523 EUR/MWh against 72 in September (§3.3), and its mean
    exceeds every later year by a factor of two (§3.1).
11. **The distribution has been narrowing from both ends since 2022** — ramp P90 down ~80 %,
    median down ~65 % (§4.4), the level band tightening (§4.5), the duration curves flattening
    (§5.2). This is the **opposite** of the team's residual-load finding, where ramps grow and the
    low tail extends. The two are not contradictory: they describe physical volatility and its
    price respectively, and nothing here identifies a cause.
12. **The negative share peaks mid-record.** On the matched window it runs 14 % → 23 % (2024) →
    11 % (§4.5), so negative imbalance prices have not simply tracked renewable build-out.

**Caveats carried forward**

13. **The final year is partial** (1 Jan – 6 Sep 2026) and the tails are seasonal (§3.7), so the
    matched window makes years comparable to each other, not to a full calendar year.
14. **§4.2's daily-seasonal strength of 0.35 overstates the case.** STL's seasonal component
    wanders over time 5.5× more than the average daily shape varies across the day; the stable
    shape is only ~118 EUR/MWh peak to trough. The team's configuration was kept for
    comparability and the caveat recorded rather than tuned away.
15. **Two sections of the team structure were deliberately skipped**, each with a reason stated
    where it belongs: federal holidays (§3.6) and the multi-series appendix items (§5).

**What this notebook does not do**

It is standalone by construction: `data/smard.csv` is never loaded, so no statement above relates
price to load, residual load or generation. The solar reading offered for the midday trough in
§3.5 and §4.2 is an interpretation of timing, **not** a tested relationship. Establishing any of
that — and the cost calculation reBAP is actually reserved for — needs the two datasets joined,
which is a separate piece of work.

### 5.3 — Closing self-check

The mechanical proof that no cell between §1 and here mutated the frame: the same row count and
span snapshotted at load time, and the same columns as `SERIES + DERIVED` — so no flag, label or
threshold column was created anywhere in the notebook.

In [ ]:
assert len(time_series) == LOADED["rows"], (len(time_series), LOADED["rows"])
assert time_series.index.min() == LOADED["start"], time_series.index.min()
assert time_series.index.max() == LOADED["end"], time_series.index.max()
assert list(time_series.columns) == SERIES + DERIVED, list(time_series.columns)
assert time_series.index.is_monotonic_increasing and time_series.index.is_unique
assert str(time_series.index.tz) == "Europe/Berlin"
assert time_series[SERIES].notna().all().all(), "a NaN appeared in the data columns"
assert (time_series[SERIES[0]] == time_series[SERIES[1]]).all()
assert YEARS == sorted(int(y) for y in time_series["year"].unique())
assert (completeness["complete_%"] == 100).sum() == len(completeness) - 1, "more than one partial month"

print("closing self-check passed")
print(f"  {len(time_series):,} rows, {LOADED['start']} -> {LOADED['end']} (unchanged since §1.4)")
print(f"  {len(time_series.columns)} columns == SERIES + DERIVED -- no flag column was created")
print(f"  {len(completeness) - 1} complete months + 1 partial (the record's last)")
print(f"  years {YEARS}")